<a href="https://www.kaggle.com/code/quddusikashaf/roformer?scriptVersionId=267532417" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Data Processing

In [1]:
# Based on the paper: "A rotary transformer cross-subject model for continuous estimation of finger joints kinematics and a transfer learning approach for new subjects"
# Dataset: Ninapro DB2 (sEMG, 12 channels, 10 joint angles)
# Architecture: Conv1D -> RoPE -> Multi-Head Attention -> Add&Norm -> Feed Forward -> Add&Norm -> FC (10 angles)
# Requirements: Download Ninapro DB2 from https://ninapro.hevs.ch/ (registration required), process using https://github.com/increase24/Ninapro-dataset-processing
# Assume processed data: sEMG windows (12 x time_steps) and labels (10 joint angles)

# Install Dependencies
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers datasets accelerate wandb tqdm numpy scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 11.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 38.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 88.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 67.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━

In [2]:
import os
import numpy as np
import glob
import scipy.io as scio
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random


In [3]:
def mu_law_normalize(x, mu=255):
    """
    Apply μ-law companding to EMG data.
    Input x should be normalized to [-1,1].
    """
    return np.sign(x) * np.log1p(mu * np.abs(x)) / np.log1p(mu)

In [4]:
def process_dataset_for_oneSubject(subject, window_size=500, stride=200, mu=255):
    print(f"Processing subject: {subject}")
    data_paths = sorted(glob.glob(f'/kaggle/input/ninapro-db2/{subject}/{subject}/*.mat'))
    
    emg_all, glove_all = [], []
    for path in data_paths[:2]:  # use first two sessions
        data = scio.loadmat(path)
        emg_all.append(data['emg'])
        glove_all.append(data['glove'])
    
    emg = np.vstack(emg_all).astype(np.float32)
    glove = np.vstack(glove_all).astype(np.float32)

    # --- EMG: μ-law normalization ---
    # Step 1: scale to [-1,1] per channel
    emg = emg / (np.max(np.abs(emg), axis=0, keepdims=True) + 1e-8)
    # Step 2: apply μ-law
    emg = mu_law_normalize(emg, mu=mu)

    # --- Glove: Min-Max normalization [0,1] per sensor ---
    glove_min = np.min(glove, axis=0, keepdims=True)
    glove_max = np.max(glove, axis=0, keepdims=True)
    glove = (glove - glove_min) / (glove_max - glove_min + 1e-8)

    # 🔧 Keep only first 10 glove joints
    cols = [1, 2, 4, 5, 7, 8, 11, 12, 15, 16]  # shifted to 0-based
    glove = glove[:, cols]

    # --- Sliding window segmentation ---
    emg_windows, glove_windows = [], []
    for i in range(0, len(emg) - window_size, stride):
        emg_win = emg[i:i+window_size, :]          # [window_size, 12]
        glove_win = glove[i+window_size-1, :]      # last frame as target
        emg_windows.append(emg_win)
        glove_windows.append(glove_win)

    emg_windows = np.array(emg_windows)            # [num_samples, window_size, 12]
    glove_windows = np.array(glove_windows)        # [num_samples, num_joints]

    # Save processed data as .npz
    output_dir = f"/kaggle/working/db2_processed/{subject}"
    os.makedirs(output_dir, exist_ok=True)
    np.savez(f"{output_dir}/data.npz", sEMG=emg_windows, angles=glove_windows)
    print(f"Saved {emg_windows.shape[0]} samples for {subject}")


In [5]:
base_path = '/kaggle/input/ninapro-db2'
subjects = sorted([folder for folder in os.listdir(base_path)])
for i, subject in enumerate(subjects[:1]):  # first 10 for testing
    process_dataset_for_oneSubject(subject)


Processing subject: DB2_s1
Saved 21806 samples for DB2_s1


In [6]:

class NinaproDataset(Dataset):
    def __init__(self, npz_files, split='train', split_ratio=0.8):
        # Load all subjects and concatenate
        sEMG_all, angles_all = [], []
        for f in npz_files:
            data = np.load(f,mmap_mode='r')
            sEMG_all.append(data['sEMG'])
            angles_all.append(data['angles'])
        self.sEMG = np.concatenate(sEMG_all, axis=0)
        self.angles = np.concatenate(angles_all, axis=0)

        # Split
        split_idx = int(split_ratio * len(self.sEMG))
        if split == 'train':
            self.sEMG = self.sEMG[:split_idx]
            self.angles = self.angles[:split_idx]
        else:
            self.sEMG = self.sEMG[split_idx:]
            self.angles = self.angles[split_idx:]

    def __len__(self):
        return len(self.sEMG)

    def __getitem__(self, idx):
        # Convert [window_size, 12] -> [12, window_size] for CNN/RNN models
        sEMG = torch.tensor(self.sEMG[idx].T, dtype=torch.float32)
        angles = torch.tensor(self.angles[idx], dtype=torch.float32)
        return sEMG, angles


In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset

class NinaproDataset(Dataset):
    def __init__(self, npz_files, split='train', split_ratio=0.8):
        sEMG_all, angles_all = [], []

        for f in npz_files:
            data = np.load(f, mmap_mode='r')
            sEMG = data['sEMG']
            angles = data['angles']

            # --- split per subject ---
            split_idx = int(split_ratio * len(sEMG))
            if split == 'train':
                sEMG_all.append(sEMG[:split_idx])
                angles_all.append(angles[:split_idx])
            else:
                sEMG_all.append(sEMG[split_idx:])
                angles_all.append(angles[split_idx:])

        # Concatenate across subjects
        self.sEMG = np.concatenate(sEMG_all, axis=0)
        self.angles = np.concatenate(angles_all, axis=0)

    def __len__(self):
        return len(self.sEMG)

    def __getitem__(self, idx):
        # Convert [window_size, 12] -> [12, window_size] for CNN/RNN models
        sEMG = torch.tensor(self.sEMG[idx].T, dtype=torch.float32)
        angles = torch.tensor(self.angles[idx], dtype=torch.float32)
        return sEMG, angles


In [7]:

# Config
class Config:
    exp_name = 'roformer_sEMG'
    data_path = '/content/processed_ninapro_db2.npz'  # Processed NPZ file (sEMG and labels)
    save_dir = 'outputs'
    max_seq_len = 400  # Time steps (50ms at 2kHz)
    num_channels = 12  # sEMG channels
    num_angles = 10  # Output joint angles
    batch_size = 32
    lr = 1e-4
    epochs = 50
    seed = 42
    num_heads = 8
    num_layers = 4
    d_model = 128
    d_ff = 512
    dropout = 0.1
    patience = 6

cfg = Config()
os.makedirs(cfg.save_dir, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(cfg.seed)
torch.manual_seed(cfg.seed)


In [8]:
processed_dir = "/kaggle/working/db2_processed"
npz_files = sorted(glob.glob(f"{processed_dir}/*/data.npz"))

train_dataset = NinaproDataset(npz_files, 'train')
test_dataset = NinaproDataset(npz_files, 'test')

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, num_workers=4 ,shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, num_workers=4, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")


Train samples: 17444, Test samples: 4362


## Model Architecture

In [9]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from tqdm import tqdm
import random
import os
from transformers import RoFormerModel, RoFormerConfig


2025-10-12 15:19:29.305942: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760282369.532943      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760282369.595530      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [10]:

# Step 2: RoPE (Rotary Position Embedding)
class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings=2048, base=10000, device=None):
        super().__init__()
        self.dim = dim
        self.max_position_embeddings = max_position_embeddings
        self.base = base
        inv_freq = 1.0 / (self.base ** (torch.arange(0, self.dim, 2).float().to(device) / self.dim))
        self.register_buffer("inv_freq", inv_freq)

        # Build here to make `torch.jit.trace` work
        self._set_cos_sin_cache(
            seq_len=max_position_embeddings, device=self.inv_freq.device, dtype=torch.get_default_dtype()
        )

    def _set_cos_sin_cache(self, seq_len, device, dtype):
        self.max_seq_len_cached = seq_len
        t = torch.arange(self.max_seq_len_cached, device=device, dtype=self.inv_freq.dtype)

        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        # Different from paper, but it uses a different permutation in order to obtain the same calculation
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos().to(dtype), persistent=False)
        self.register_buffer("sin_cached", emb.sin().to(dtype), persistent=False)

    def forward(self, x, seq_len=None):
        # x: [bs, num_attention_heads, seq_len, head_size]
        if seq_len > self.max_seq_len_cached:
            self._set_cos_sin_cache(seq_len=seq_len, device=x.device, dtype=x.dtype)
        return (
            self.cos_cached[:seq_len].to(dtype=x.dtype),
            self.sin_cached[:seq_len].to(dtype=x.dtype),
        )


In [11]:

def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

# def apply_rotary_pos_emb(q, k, cos, sin, position_ids=None):
#     # The rotary dim starts after the `max_position_embeddings` and `offset` if any.
#     gather_indices = torch.arange(q.shape[-2], device=q.device).unsqueeze(0).unsqueeze(0).unsqueeze(-1).repeat(q.shape[0], q.shape[1], 1, q.shape[3])
#     if position_ids is not None:
#         gather_indices = gather_indices + position_ids.unsqueeze(1).unsqueeze(2)
#     cos = cos[gather_indices].squeeze(-1)
#     sin = sin[gather_indices].squeeze(-1)
#     print("q:", q.shape, "cos:", cos.shape)
#     q_embed = (q * cos) + (rotate_half(q) * sin)
#     k_embed = (k * cos) + (rotate_half(k) * sin)
#     return q_embed, k_embed
def apply_rotary_pos_emb(q, k, cos, sin):
    # q, k: [batch, seq_len, d_model]
    # cos, sin: [seq_len, d_model]
    cos = cos.unsqueeze(0)  # [1, seq_len, d_model]
    sin = sin.unsqueeze(0)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed



In [12]:
class RoFormerLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.activation = nn.GELU()
    def forward(self, src, cos=None, sin=None, src_mask=None, src_key_padding_mask=None):
        q = k = src
        if cos is not None and sin is not None:
            q, k = apply_rotary_pos_emb(q, k, cos, sin)  # no unsqueeze
        src2 = self.self_attn(q, k, src, attn_mask=src_mask, key_padding_mask=src_key_padding_mask)[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src

    # def forward(self, src, cos=None, sin=None, src_mask=None, src_key_padding_mask=None):
    #     # Apply attention
    #     q = k = src
    #     if cos is not None and sin is not None:
    #         q, k = apply_rotary_pos_emb(q.unsqueeze(1), k.unsqueeze(1), cos, sin)
    #         q, k = q.squeeze(1), k.squeeze(1)

    #     src2 = self.self_attn(q, k, src, attn_mask=src_mask, key_padding_mask=src_key_padding_mask)[0]
    #     src = src + self.dropout1(src2)
    #     src = self.norm1(src)
    #     src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
    #     src = src + self.dropout2(src2)
    #     src = self.norm2(src)
    #     return src


In [13]:

class RoFormer(nn.Module):
    def __init__(self, num_channels=12, time_steps=100, num_layers=4, num_heads=8, d_model=128, d_ff=512, num_angles=10, dropout=0.1):
        super().__init__()
        self.conv1d = nn.Conv1d(num_channels, d_model, kernel_size=3, padding=1)  # Conv1D on sEMG channels
        self.rope = RotaryEmbedding(d_model)
        self.layers = nn.ModuleList([RoFormerLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.fc = nn.Linear(d_model, num_angles)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: [batch, channels, time_steps]
        x = self.conv1d(x)                # [batch, d_model, time_steps]
        x = x.transpose(1, 2)             # [batch, time_steps, d_model]
        x = self.dropout(x)
    
        seq_len = x.shape[1]
        cos, sin = self.rope(x, seq_len=seq_len)
    
        for layer in self.layers:
            x = layer(x, cos, sin)
    
        x = x.mean(dim=1)
        x = self.fc(x)
        return x


# Step 4: Training Loop
model = RoFormer(cfg.num_channels, cfg.max_seq_len, cfg.num_layers, cfg.num_heads, cfg.d_model, cfg.d_ff, cfg.num_angles, cfg.dropout).to(device)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=cfg.lr)
criterion = nn.MSELoss()


Using 2 GPUs!


## Training

In [14]:

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch_sEMG, batch_angles in tqdm(loader):
        batch_sEMG = batch_sEMG.to(device)
        batch_angles = batch_angles.to(device)
        optimizer.zero_grad()
        outputs = model(batch_sEMG)
        loss = criterion(outputs, batch_angles)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch_sEMG, batch_angles in loader:
            batch_sEMG = batch_sEMG.to(device)
            batch_angles = batch_angles.to(device)
            outputs = model(batch_sEMG)
            loss = criterion(outputs, batch_angles)
            total_loss += loss.item()
            all_preds.append(outputs.cpu().numpy())
            all_targets.append(batch_angles.cpu().numpy())
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    rmse = np.sqrt(mean_squared_error(all_targets, all_preds))
    r2 = r2_score(all_targets, all_preds)
    return total_loss / len(loader), rmse, r2


In [15]:
def model_size(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # Assume FP32 unless you cast model to FP16/BF16
    bytes_per_param = 4  

    total_size_mb = total_params * bytes_per_param / (1024 ** 2)
    trainable_size_mb = trainable_params * bytes_per_param / (1024 ** 2)

    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Total size (MB): {total_size_mb:.2f}")
    print(f"Trainable size (MB): {trainable_size_mb:.2f}")

    return total_params, total_size_mb
model_size(model)

Total parameters: 799,114
Trainable parameters: 799,114
Total size (MB): 3.05
Trainable size (MB): 3.05


(799114, 3.0483779907226562)

In [16]:

# Training
best_r = 0.0
patience_counter = 0
for epoch in range(cfg.epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, rmse, r2 = validate(model, test_loader, criterion, device)
    print(f'Epoch {epoch+1}/{cfg.epochs}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}')
    if rmse > best_r:
        best_r = rmse
        torch.save(model.state_dict(), os.path.join(cfg.save_dir, 'best_roformer.pt'))
        patience_counter = 0
    # else:
    #     patience_counter += 1
    #     if patience_counter >= cfg.patience:
    #         print(f'Early stopping at epoch {epoch+1}')
    #         break



100%|██████████| 546/546 [00:48<00:00, 11.25it/s]


Epoch 1/50: Train Loss=0.0235, Val Loss=0.0227, RMSE=0.1503, R2=-0.2587


100%|██████████| 546/546 [00:48<00:00, 11.37it/s]


Epoch 2/50: Train Loss=0.0159, Val Loss=0.0182, RMSE=0.1349, R2=-0.0442


100%|██████████| 546/546 [00:50<00:00, 10.76it/s]


...

Epoch 22/50: Train Loss=0.0061, Val Loss=0.0208, RMSE=0.1440, R2=-0.2562


100%|██████████| 546/546 [00:53<00:00, 10.26it/s]


Epoch 23/50: Train Loss=0.0058, Val Loss=0.0219, RMSE=0.1474, R2=-0.2484


 68%|██████▊   | 372/546 [00:36<00:16, 10.33it/s]


KeyboardInterrupt: 

In [ ]:
# Inference Example
model.load_state_dict(torch.load(os.path.join(cfg.save_dir, 'best_roformer.pt')))
model.eval()


In [ ]:

def predict_angles(sEMG_window):  # sEMG_window: [1, 12, 100]
    sEMG_tensor = torch.tensor(sEMG_window, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        angles = model(sEMG_tensor)
    return angles.cpu().numpy()


In [ ]:
def predict_and_compare(sEMG_window, true_angles):  
    """
    sEMG_window: [12, time_steps] or [1, 12, time_steps]
    true_angles: [num_joints] ground-truth glove values
    """
    # Ensure correct shape: [batch, 12, time_steps]
    if sEMG_window.ndim == 2:   # [12, time_steps]
        sEMG_tensor = torch.tensor(sEMG_window, dtype=torch.float32).unsqueeze(0).to(device)
    elif sEMG_window.ndim == 3: # [1, 12, time_steps]
        sEMG_tensor = torch.tensor(sEMG_window, dtype=torch.float32).to(device)
    else:
        raise ValueError("Unexpected sEMG_window shape")

    with torch.no_grad():
        pred_angles = model(sEMG_tensor)

    pred_angles = pred_angles.cpu().numpy().squeeze()
    true_angles = np.array(true_angles)

    # Print comparison
    print("True angles:      ", np.round(true_angles, 3))
    print("Predicted angles: ", np.round(pred_angles, 3))

    return pred_angles


In [ ]:
# Take one batch from DataLoader
for sEMG_batch, angles_batch in train_loader:
    sEMG_sample = sEMG_batch[0].unsqueeze(0).numpy()   # [1, 12, time_steps]
    true_sample = angles_batch[0].numpy()              # [num_joints]
    predict_and_compare(sEMG_sample, true_sample)
    break
